In [2]:
MOODLE_URL = "http://moodle-2025-2.lerdo.tecnm.mx/moodle/webservice/rest/server.php"
MOODLE_TOKEN = "9238b326cd606cf2b57eba793567a740"
MOODLE_COURSE_ID = 180

OUTDIR = "/content/data_foros"

import os
os.makedirs(OUTDIR, exist_ok=True)

import requests
import pandas as pd

def call(wsfunction, **params):
    payload = {
        "wstoken": MOODLE_TOKEN,
        "wsfunction": wsfunction,
        "moodlewsrestformat": "json",
    }
    payload.update(params)

    r = requests.post(MOODLE_URL, data=payload, timeout=60)
    r.raise_for_status()
    data = r.json()

    if isinstance(data, dict) and data.get("exception"):
        raise RuntimeError(
            f"Error WS [{wsfunction}] {data.get('errorcode')}: {data.get('message')}"
        )

    return data

info = call("core_webservice_get_site_info")
print("Conectado como:", info.get("fullname"))

usuarios = call("core_enrol_get_enrolled_users", courseid=MOODLE_COURSE_ID)

users_df = pd.DataFrame([{
    "user_id": u.get("id"),
    "fullname": u.get("fullname"),
    "username": u.get("username"),
    "email": u.get("email"),
    "suspended": u.get("suspended", 0),
} for u in usuarios])

print("Usuarios:", len(users_df))

foros = call("mod_forum_get_forums_by_courses", **{"courseids[0]": MOODLE_COURSE_ID})

forums_df = pd.DataFrame([{
    "forum_id": f.get("id"),
    "forum_name": f.get("name"),
    "forum_type": f.get("type"),
} for f in foros])

print("Foros:", len(forums_df))

rows_discussions = []

for _, foro in forums_df.iterrows():
    forum_id = foro["forum_id"]

    try:
        data = call("mod_forum_get_forum_discussions", forumid=forum_id)
    except Exception as e:
        print(f"Error en foro {forum_id}: {e}")
        continue

    if isinstance(data, dict):
        data = data.get("discussions", [])

    for d in data:
        rows_discussions.append({
            "discussion_id": d.get("discussion"),
            "forum_id": forum_id,
            "forum_name": foro["forum_name"],
            "user_id": d.get("userid") if d.get("userid") is not None else d.get("author", {}).get("id"),
            "discussion_name": d.get("name"),
        })

discussions_df = pd.DataFrame(rows_discussions)
print("Discusiones:", len(discussions_df))

rows_posts = []

for _, d in discussions_df.iterrows():
    discussion_id = d["discussion_id"]

    try:
        data = call("mod_forum_get_discussion_posts", discussionid=discussion_id)
    except Exception as e:
        print(f"Error en discusión {discussion_id}: {e}")
        continue

    if isinstance(data, dict):
        data = data.get("posts", [])

    for p in data:
        rows_posts.append({
            "post_id": p.get("id"),
            "discussion_id": discussion_id,
            "forum_id": d["forum_id"],
            "forum_name": d["forum_name"],
            "user_id": p.get("author", {}).get("id"),
            "parent_id": p.get("parentid"),
            "subject": p.get("subject"),
            "created": p.get("created"),
        })

posts_df = pd.DataFrame(rows_posts)
print("Posts:", len(posts_df))

if not posts_df.empty:
    posts_df["created"] = pd.to_datetime(posts_df["created"], unit="s", errors="coerce")
    posts_df["is_reply"] = posts_df["parent_id"].fillna(0).astype(int) != 0
    posts_df["date"] = posts_df["created"].dt.date
else:
    posts_df = pd.DataFrame(columns=[
        "post_id","discussion_id","forum_id","forum_name",
        "user_id","parent_id","subject","created","is_reply","date"
    ])

if not posts_df.empty:
    indicator_df = (
        posts_df.groupby("user_id")
        .agg(
            total_posts=("post_id", "count"),
            replies=("is_reply", "sum"),
            discussions_participated=("discussion_id", "nunique"),
            forums_participated=("forum_id", "nunique"),
            active_days=("date", "nunique"),
        )
        .reset_index()
    )
else:
    indicator_df = pd.DataFrame(columns=[
        "user_id","total_posts","replies",
        "discussions_participated","forums_participated","active_days"
    ])

indicator_df["root_posts"] = indicator_df["total_posts"] - indicator_df["replies"]

final_df = users_df.merge(indicator_df, on="user_id", how="left")

cols = [
    "total_posts","replies","root_posts",
    "discussions_participated","forums_participated","active_days"
]

for c in cols:
    final_df[c] = final_df[c].fillna(0).astype(int)

users_df.to_csv(f"{OUTDIR}/users.csv", index=False)
forums_df.to_csv(f"{OUTDIR}/forums.csv", index=False)
discussions_df.to_csv(f"{OUTDIR}/discussions.csv", index=False)
posts_df.to_csv(f"{OUTDIR}/posts.csv", index=False)
final_df.to_csv(f"{OUTDIR}/forum_interaction_indicator.csv", index=False)

print("\nProceso terminado.")
print(final_df.head())

Conectado como: KARLA VERONICA RODRIGUEZ LOZANO
Usuarios: 60
Foros: 3
Discusiones: 2
Posts: 3

Proceso terminado.
   user_id                         fullname   username  \
0      116  KARLA VERONICA RODRIGUEZ LOZANO   itsl0810   
1     1147         HUGO CÉSAR ALVARADO VELA   itsl0304   
2     1168    ANA FERNANDA ACOSTA SIFUENTES  212310246   
3     1184  VICTOR SILVERIO AGUIRRE SANCHEZ  222310529   
4     1217         JOSE ANGEL ALVA ESPINOZA  222310665   

                             email  suspended  total_posts  replies  \
0         karla.rl@itslerdo.edu.mx          0            3        1   
1  div_informatica@itslerdo.edu.mx          0            0        0   
2        212310246@itslerdo.edu.mx          0            0        0   
3        222310529@itslerdo.edu.mx          0            0        0   
4        222310665@itslerdo.edu.mx          0            0        0   

   discussions_participated  forums_participated  active_days  root_posts  
0                         2       

In [ ]:
!zip -r resultados.zip data data_foros

In [ ]:
from google.colab import files
files.download("resultados.zip")